# DS 6021 Final Project: API price-availability test

**Question:** how many concert events come back from the Ticketmaster (and optionally SeatGeek) API *with a price*?

**Setup:** put a file named `.env` in the same folder as this notebook containing:
```
TM_API_KEY=your_ticketmaster_consumer_key
SG_CLIENT_ID=your_seatgeek_client_id   # optional
```
Add `.env` to `.gitignore` so keys never reach GitHub. Then run the cells top to bottom (Run All).

Uses about 60 Ticketmaster calls (daily limit is 5,000).

*AI disclosure: this notebook was drafted with Claude (Anthropic) and should be flagged as AI-generated in the repo, per the course GenAI policy.*

## 1. Install packages (first run only)

In [1]:
%pip install requests python-dotenv pandas --quiet

Note: you may need to restart the kernel to use updated packages.


## 2. Load keys and settings

In [2]:
import os
import time
from collections import Counter

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()  # reads .env from the notebook's folder
TM_KEY = os.getenv("TM_API_KEY")
SG_ID = os.getenv("SG_CLIENT_ID")

print("Ticketmaster key found:", bool(TM_KEY))
print("SeatGeek ID found:     ", bool(SG_ID))

TM_URL = "https://app.ticketmaster.com/discovery/v2/events.json"
TM_DETAIL_URL = "https://app.ticketmaster.com/discovery/v2/events/{}.json"
SG_URL = "https://api.seatgeek.com/2/events"

# DMV first, then large markets for comparison
STATES = ["DC", "MD", "VA", "NY", "CA", "TX", "IL", "FL", "GA", "TN"]
DMV = {"DC", "MD", "VA"}
PAGE_SIZE = 200   # Ticketmaster max per page
MAX_PAGES = 5     # Ticketmaster stops at ~1,000 results per query
PAUSE = 0.3       # seconds between calls

Ticketmaster key found: True
SeatGeek ID found:      False


## 3. Helper functions

In [3]:
def get(url, params):
    """GET with basic error handling; returns JSON dict or None."""
    try:
        r = requests.get(url, params=params, timeout=20)
    except requests.RequestException as e:
        print(f"  request failed: {e}")
        return None
    if r.status_code == 401:
        raise RuntimeError("401 Unauthorized: check the API key in your .env file.")
    if r.status_code == 429:
        print("  hit rate limit; waiting 2 seconds")
        time.sleep(2)
        return get(url, params)
    if r.status_code != 200:
        print(f"  HTTP {r.status_code}: {r.text[:150]}")
        return None
    return r.json()


def tm_flatten(ev):
    """Pull the fields we care about out of one Ticketmaster event."""
    venue = (ev.get("_embedded", {}).get("venues") or [{}])[0]
    cls = (ev.get("classifications") or [{}])[0]
    acts = ev.get("_embedded", {}).get("attractions") or []
    pr = (ev.get("priceRanges") or [{}])[0]
    return {
        "event_id": ev.get("id"),
        "name": ev.get("name"),
        "date": ev.get("dates", {}).get("start", {}).get("localDate"),
        "local_time": ev.get("dates", {}).get("start", {}).get("localTime"),
        "status": ev.get("dates", {}).get("status", {}).get("code"),
        "onsale_date": ev.get("sales", {}).get("public", {}).get("startDateTime"),
        "venue": venue.get("name"),
        "city": venue.get("city", {}).get("name"),
        "state": venue.get("state", {}).get("stateCode"),
        "genre": cls.get("genre", {}).get("name"),
        "subgenre": cls.get("subGenre", {}).get("name"),
        "headliner": acts[0].get("name") if acts else None,
        "n_acts": len(acts),
        "promoter": (ev.get("promoter") or {}).get("name"),
        "ticket_limit": (ev.get("ticketLimit") or {}).get("info"),
        "price_min": pr.get("min"),
        "price_max": pr.get("max"),
        "price_currency": pr.get("currency"),
        "has_price": bool(ev.get("priceRanges")),
    }

## 4. Pull Ticketmaster music events by state
Takes about a minute.

In [4]:
rows, seen = [], set()
for st in STATES:
    n_state = 0
    for page in range(MAX_PAGES):
        params = {
            "apikey": TM_KEY,
            "classificationName": "music",
            "countryCode": "US",
            "stateCode": st,
            "size": PAGE_SIZE,
            "page": page,
            "sort": "date,asc",
        }
        data = get(TM_URL, params)
        time.sleep(PAUSE)
        if not data or "_embedded" not in data:
            break
        for ev in data["_embedded"]["events"]:
            if ev["id"] not in seen:
                seen.add(ev["id"])
                rows.append(tm_flatten(ev))
                n_state += 1
        if page + 1 >= data.get("page", {}).get("totalPages", 0):
            break
    print(f"{st}: {n_state} events")

print(f"\nTotal unique music events: {len(rows)}")

DC: 377 events
MD: 225 events
VA: 499 events
NY: 1000 events
CA: 1000 events
TX: 1000 events
IL: 1000 events
FL: 1000 events
GA: 839 events
TN: 1000 events

Total unique music events: 7940


## 5. Check unpriced events one at a time
Some APIs leave prices out of search results but include them on the single-event page, so this checks 25 unpriced events individually.

In [5]:
unpriced = [r for r in rows if not r["has_price"]][:25]
found_in_detail = 0
for r in unpriced:
    data = get(TM_DETAIL_URL.format(r["event_id"]), {"apikey": TM_KEY})
    time.sleep(PAUSE)
    if data and data.get("priceRanges"):
        found_in_detail += 1
        pr = data["priceRanges"][0]
        r.update(price_min=pr.get("min"), price_max=pr.get("max"),
                 price_currency=pr.get("currency"), has_price=True)

print(f"Extra prices found by checking {len(unpriced)} events individually: {found_in_detail}")

Extra prices found by checking 25 events individually: 0


## 6. Results

In [6]:
tm = pd.DataFrame(rows)
tm.to_csv("tm_test_events.csv", index=False)

n = len(tm)
n_priced = tm["has_price"].sum()
dmv = tm[tm["state"].isin(DMV)]

print(f"Total events:        {n}")
print(f"Events WITH a price: {n_priced} ({n_priced / n:.1%})")
print(f"DMV events:          {len(dmv)}  (priced: {dmv['has_price'].sum()})")
print()
print("Rule of thumb: 1,000+ priced events -> keep the event-level plan;")
print("far fewer -> switch to Pollstar/Billboard tour data for the target.")

Total events:        7940
Events WITH a price: 2045 (25.8%)
DMV events:          1101  (priced: 62)

Rule of thumb: 1,000+ priced events -> keep the event-level plan;
far fewer -> switch to Pollstar/Billboard tour data for the target.


In [7]:
# Priced share by state
tm.groupby("state")["has_price"].agg(events="size", priced="sum", share_priced="mean") \
  .sort_values("events", ascending=False).style.format({"share_priced": "{:.0%}"})

,events,priced,share_priced
state,,,
CA,1000,306,31%
FL,1000,277,28%
TX,1000,57,6%
IL,1000,428,43%
TN,1000,428,43%
NY,1000,478,48%
GA,839,9,1%
VA,499,29,6%
DC,377,29,8%


In [8]:
# How often each feature is filled in
cols = ["genre", "subgenre", "headliner", "promoter", "onsale_date",
        "ticket_limit", "venue", "state", "price_min", "price_max"]
filled = tm[cols].replace({"Undefined": None, "": None}).notna().mean()
filled.to_frame("share_filled").style.format("{:.0%}")

,share_filled
genre,97%
subgenre,93%
headliner,81%
promoter,50%
onsale_date,100%
ticket_limit,36%
venue,100%
state,100%
price_min,26%
price_max,26%


In [9]:
print("Top genres:")
tm["genre"].value_counts().head(8)

Top genres:


genre
Rock                1968
Other               1209
Country              664
Pop                  636
Alternative          443
Jazz                 430
Hip-Hop/Rap          413
Dance/Electronic     357
Name: count, dtype: int64

In [10]:
# Sample of priced events (empty table = no prices came back)
tm.loc[tm["has_price"], ["date", "headliner", "venue", "state", "genre", "price_min", "price_max"]].head(15)

,date,headliner,venue,state,genre,price_min,price_max
13,2026-10-22,NaN,Howard Theatre,DC,Religious,33.21,72.35
17,2026-10-23,Low Cut Connie,Howard Theatre,DC,Rock,33.21,33.21
23,2026-10-23,NaN,Howard Theatre,DC,Hip-Hop/Rap,20.00,54.84
54,2026-10-29,Eivor,Howard Theatre,DC,Pop,36.37,87.87
61,2026-10-30,Boy Harsher,Howard Theatre,DC,Dance/Electronic,0.00,0.00
77,2026-11-01,Eva Ayllon,Howard Theatre,DC,World,116.64,116.64
104,2026-11-08,NaN,Karma DC Live Music Venue,DC,Pop,58.97,277.33
105,2026-11-08,Akeem Ali,Howard Theatre,DC,Hip-Hop/Rap,33.28,102.29
113,2026-11-10,NaN,Howard Theatre,DC,Pop,74.93,120.25
119,2026-11-11,NaN,Howard Theatre,DC,Folk,0.00,0.00


## 7. Optional: SeatGeek test
Skips automatically if `SG_CLIENT_ID` isn't in `.env`.

In [6]:
import os, time
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(".env", override=True)
SG_CLIENT_ID = os.getenv("SG_CLIENT_ID")
SG_APP_SECRET = os.getenv("SG_APP_SECRET")   # only if your .env has one
SG_URL = "https://api.seatgeek.com/2/events"
PAUSE = 0.3

def get(url, params):
    r = requests.get(url, params=params, timeout=20)
    if r.status_code != 200:
        print(f"HTTP {r.status_code}: {r.text[:200]}")
        return None
    return r.json()

print("SeatGeek ID found:", bool(SG_CLIENT_ID))

# ---

if not SG_CLIENT_ID:
    print("Skipping SeatGeek: no SG_CLIENT_ID set.")
else:
    sg_rows = []
    for st in ["DC", "MD", "VA", "NY", "CA"]:
        data = get(SG_URL, {"client_id": SG_CLIENT_ID, "taxonomies.name": "concert",
                    "venue.state": st, "per_page": 100})
        time.sleep(PAUSE)
        if not data:
            continue
        for ev in data.get("events", []):
            stats = ev.get("stats") or {}
            v = ev.get("venue") or {}
            perf = (ev.get("performers") or [{}])[0]
            sg_rows.append({
                "event_id": ev.get("id"), "title": ev.get("title"),
                "date": ev.get("datetime_local"), "venue": v.get("name"),
                "state": v.get("state"), "venue_capacity": v.get("capacity"),
                "performer": perf.get("name"), "event_score": ev.get("score"),
                "lowest_price": stats.get("lowest_price"),
                "average_price": stats.get("average_price"),
                "highest_price": stats.get("highest_price"),
                "listing_count": stats.get("listing_count"),
            })
        print(f"{st}: {len(data.get('events', []))} events")

    sg = pd.DataFrame(sg_rows)
    sg.to_csv("sg_test_events.csv", index=False)
    check = ["lowest_price", "average_price", "listing_count", "venue_capacity", "event_score"]
    display(sg[check].replace(0, None).notna().mean().to_frame("share_filled").style.format("{:.0%}"))

SeatGeek ID found: True
DC: 100 events
MD: 100 events
VA: 100 events
NY: 100 events
CA: 100 events


,share_filled
lowest_price,0%
average_price,0%
listing_count,0%
venue_capacity,25%
event_score,100%
